# Module 10 — OpenSearch (recherche BM25)

Indexer le silver et **tester plusieurs types de requêtes** pour comprendre ce que BM25 fait bien — et ce qu'il ne fait pas.

**Prérequis** : `docker compose up -d opensearch`, `OPENSEARCH_URL` dans `.env`, corpus indexé (`uv run presslake index`).

Tuto : [`docs/modules/10-opensearch.md`](../docs/modules/10-opensearch.md)

## Étape 1 — Santé du cluster

`status: yellow` en single-node sans réplica = **normal** en dev.

In [1]:
from presslake.search.client import get_opensearch_client
from presslake.search.index import count_documents

client = get_opensearch_client()
health = client.cluster.health()
print("status:", health["status"])
print("documents indexés:", count_documents(client))

status: yellow
documents indexés: 147


## Étape 2 — Indexer (si besoin)

```bash
uv run presslake index        # tous les articles status=parsed
uv run presslake index --limit 10
```

Relance la cellule 1 pour vérifier que `documents indexés` > 0.

## Étape 3 — Helper d'affichage

Chaque cas de test réutilise `show_search()` :
- **score** : pertinence BM25 (relatif au corpus, pas un %)
- **snippet** : extrait avec `<em>terme</em>` = surlignage OpenSearch

In [2]:
import re

from presslake.search.client import get_opensearch_client
from presslake.search.index import search_articles


def _clean_snippet(snippet: str, max_len: int = 100) -> str:
    """Retire les balises HTML du highlight pour la lecture."""
    text = re.sub(r"</?em>", "", snippet or "")
    text = " ".join(text.split())
    return text[:max_len] + ("…" if len(text) > max_len else "")


def show_search(query: str, *, limit: int = 5, lang: str | None = None, note: str = "") -> list[dict]:
    """
    Lance une requête BM25 et affiche les hits.

    Returns:
        Liste brute des hits (pour inspection dans le notebook).
    """
    hits = search_articles(get_opensearch_client(), query, limit=limit, lang=lang)
    lang_hint = f" (lang={lang})" if lang else ""
    print(f"Requête : {query!r}{lang_hint}")
    if note:
        print(f"Note    : {note}")
    print(f"Résultats : {len(hits)}\n")

    if not hits:
        print("→ Aucun hit (terme absent du corpus ou index vide).")
        return hits

    for i, hit in enumerate(hits, 1):
        title = (hit.get("title") or "(sans titre)")[:70]
        print(f"{i}. [{hit['score']:.2f}] {hit.get('feed_id')} | {title}")
        snippet = _clean_snippet(hit.get("snippet", ""))
        if snippet:
            print(f"   … {snippet}")
    print()
    return hits

### Cas A — Mot rare / nom de lieu exact

**Attendu** : BM25 excellent. Le terme est peu fréquent → score élevé, articles sur le Népal en tête.

C'est le cas d'usage principal du module 10.

In [3]:
hits_nepal = show_search(
    "Népal",
    note="Mot rare dans le corpus → forte pertinence",
)

Requête : 'Népal'
Note    : Mot rare dans le corpus → forte pertinence
Résultats : 5

1. [7.94] france24 | Crise au Népal : onde de choc régionale ?
   … Crise au Népal : onde de choc régionale ?
2. [7.34] france24 | Désastre au Népal : les intox inondent les réseaux sociaux
   … Désastre au Népal : les intox inondent les réseaux sociaux Pour afficher ce contenu YouTube, il est …
3. [7.07] france24 | 🇳🇵 Népal : après la crue meurtrière, survivre coûte que coûte
   … . 🇳🇵 Népal : après la crue meurtrière, survivre coûte que coûte Publié le : Partager Après la crue m…
4. [6.38] lemonde-une | Au Népal, après les inondations, les glaciologues sidérés par l’ampleu
   … Au Népal, après les inondations, les glaciologues sidérés par l’ampleur de la catastrophe
5. [5.98] rfi | Crue dans l'Himalaya: le Népal commence les enterrements de masse de v
   … Crue dans l'Himalaya: le Népal commence les enterrements de masse de victimes impossibles à identifi…



### Cas B — Nom propre (personnalité)

**Attendu** : l'article sur cette personne remonte en #1. BM25 gère bien les noms propres **s'ils sont dans le texte indexé**.

In [4]:
hits_balladur = show_search(
    "Balladur",
    note="Nom propre — doit cibler l'obituary / article dédié",
)

Requête : 'Balladur'
Note    : Nom propre — doit cibler l'obituary / article dédié
Résultats : 3

1. [8.68] rfi | France: l'ancien Premier ministre Édouard Balladur est mort
   … Ce que laisse Édouard Balladur, c'est d'abord le souvenir d'un chef de gouvernement.
2. [7.50] lemonde-une | Edouard Balladur, ancien premier ministre, est mort à l’âge de 97 ans
   … Edouard Balladur, ancien premier ministre, est mort à l’âge de 97 ans
3. [7.25] france24 | 🔴 L'ancien Premier ministre Édouard Balladur est mort à l'âge de 97 an
   … Mort de l'ancien Premier ministre Édouard Balladur à 97 ans Édouard Balladur, Premier ministre entre…



### Cas C — Requête multi-mots (recoupement)

**Attendu** : les articles contenant **plusieurs** termes de la requête montent. Utile pour affiner « Népal » + « inondations ».

In [5]:
hits_multi = show_search(
    "Népal inondations glaciologues",
    note="Multi-mots — score plus discriminant qu'un seul mot vague",
)

Requête : 'Népal inondations glaciologues'
Note    : Multi-mots — score plus discriminant qu'un seul mot vague
Résultats : 5

1. [24.16] lemonde-une | Au Népal, après les inondations, les glaciologues sidérés par l’ampleu
   … Au Népal, après les inondations, les glaciologues sidérés par l’ampleur de la catastrophe
2. [7.94] france24 | Crise au Népal : onde de choc régionale ?
   … Crise au Népal : onde de choc régionale ?
3. [7.34] france24 | Désastre au Népal : les intox inondent les réseaux sociaux
   … Désastre au Népal : les intox inondent les réseaux sociaux Pour afficher ce contenu YouTube, il est …
4. [7.07] france24 | 🇳🇵 Népal : après la crue meurtrière, survivre coûte que coûte
   … . 🇳🇵 Népal : après la crue meurtrière, survivre coûte que coûte Publié le : Partager Après la crue m…
5. [5.98] rfi | Crue dans l'Himalaya: le Népal commence les enterrements de masse de v
   … Crue dans l'Himalaya: le Népal commence les enterrements de masse de victimes impossibles à identifi…



### Cas D — Mot très fréquent (bruit)

**Attendu** : beaucoup de hits, scores **plus bas** et moins discriminants. BM25 pénalise un peu la fréquence, mais « France » reste partout dans un flux France 24.

→ Utile pour voir pourquoi on ne se contente pas d'un `LIKE '%France%'`.

In [6]:
hits_france = show_search(
    "France",
    limit=3,
    note="Terme fréquent — pertinence moins nette",
)

Requête : 'France'
Note    : Terme fréquent — pertinence moins nette
Résultats : 3

1. [8.68] rfi | France: l'ancien Premier ministre Édouard Balladur est mort
   … France: l'ancien Premier ministre Édouard Balladur est mort L’ex-Premier ministre français (1993-199…
2. [7.50] rfi | Rentrée scolaire: le salaire des enseignants en France au cœur des déb
   … Rentrée scolaire: le salaire des enseignants en France au cœur des débats Les enseignants français e…
3. [7.02] rfi | France: sous pression des oppositions, Sébastien Lecornu lance les dis
   … France: sous pression des oppositions, Sébastien Lecornu lance les discussions sur le budget 2027 Le…



### Cas E — Paraphrase (limite du BM25)

**Attendu** : résultats **moins pertinents** ou vides si aucun article ne contient ces mots exacts.

Ex. : « catastrophe himalayenne » sans écrire « Népal ». C'est la limite du lexical — le module 11 (Qdrant) adresse ce cas par similarité sémantique.

In [7]:
hits_para = show_search(
    "catastrophe himalayenne crue meurtrière",
    note="Paraphrase — BM25 cherche des MOTS, pas du sens",
)

# Compare avec le cas A : même sujet, mot exact
print("--- Comparaison ---")
print(f"Cas A (Népal)      : {len(hits_nepal)} hits, top score = {hits_nepal[0]['score']:.2f}" if hits_nepal else "Cas A : 0 hits")
print(f"Cas E (paraphrase) : {len(hits_para)} hits", end="")
if hits_para:
    print(f", top score = {hits_para[0]['score']:.2f}")
else:
    print()

Requête : 'catastrophe himalayenne crue meurtrière'
Note    : Paraphrase — BM25 cherche des MOTS, pas du sens
Résultats : 5

1. [15.48] france24 | 🇳🇵 Népal : après la crue meurtrière, survivre coûte que coûte
   … . 🇳🇵 Népal : après la crue meurtrière, survivre coûte que coûte Publié le : Partager Après la crue m…
2. [15.48] rfi | États-Unis: crue brutale et meurtrière dans le Grand Canyon
   … États-Unis: crue brutale et meurtrière dans le Grand Canyon Dans l'ouest des États-Unis, une crue so…
3. [8.89] lemonde-une | Au Népal, après les inondations, les glaciologues sidérés par l’ampleu
   … Au Népal, après les inondations, les glaciologues sidérés par l’ampleur de la catastrophe
4. [8.87] france24 | Désastre au Népal : les intox inondent les réseaux sociaux
   … 1 min 5 jours après la crue
5. [8.45] rfi | Crue dans l'Himalaya: le Népal commence les enterrements de masse de v
   … Crue dans l'Himalaya: le Népal commence les enterrements de masse de victimes impossibles à identifi…

--

### Cas F — Terme introuvable

**Attendu** : 0 résultat. Comportement normal — pas d'hallucination possible côté index.

In [8]:
show_search(
    "xyztokenintrouvable123",
    note="Token absent — doit retourner 0 hit",
)

Requête : 'xyztokenintrouvable123'
Note    : Token absent — doit retourner 0 hit
Résultats : 0

→ Aucun hit (terme absent du corpus ou index vide).


[]

### Cas G — Filtre langue (`--lang`)

**Attendu** : seuls les articles `content_lang=fr` (ou `en`) remontent, avec l'analyzer dédié (ADR 0003).

Compare `Népal` sans filtre vs `--lang fr` vs `--lang en`.

In [9]:
hits_nepal_all = show_search("Népal", note="Sans filtre langue")
hits_nepal_fr = show_search("Népal", lang="fr", note="Corpus FR uniquement")
hits_nepal_en = show_search("Nepal", lang="en", note="Corpus EN — souvent 0 si pas d'article EN sur le sujet")

print("--- Comparaison ---")
print(f"Sans filtre : {len(hits_nepal_all)} hits")
print(f"--lang fr   : {len(hits_nepal_fr)} hits")
print(f"--lang en   : {len(hits_nepal_en)} hits")

Requête : 'Népal'
Note    : Sans filtre langue
Résultats : 5

1. [7.94] france24 | Crise au Népal : onde de choc régionale ?
   … Crise au Népal : onde de choc régionale ?
2. [7.34] france24 | Désastre au Népal : les intox inondent les réseaux sociaux
   … Désastre au Népal : les intox inondent les réseaux sociaux Pour afficher ce contenu YouTube, il est …
3. [7.07] france24 | 🇳🇵 Népal : après la crue meurtrière, survivre coûte que coûte
   … . 🇳🇵 Népal : après la crue meurtrière, survivre coûte que coûte Publié le : Partager Après la crue m…
4. [6.38] lemonde-une | Au Népal, après les inondations, les glaciologues sidérés par l’ampleu
   … Au Népal, après les inondations, les glaciologues sidérés par l’ampleur de la catastrophe
5. [5.98] rfi | Crue dans l'Himalaya: le Népal commence les enterrements de masse de v
   … Crue dans l'Himalaya: le Népal commence les enterrements de masse de victimes impossibles à identifi…

Requête : 'Népal' (lang=fr)
Note    : Corpus FR uniquement
Résulta

## Étape 4 — Synthèse

| Cas | Requête type | BM25 |
|---|---|---|
| A | Mot rare exact (`Népal`) | ✅ Fort |
| B | Nom propre (`Balladur`) | ✅ Fort |
| C | Multi-mots | ✅ Bon filtre |
| D | Mot fréquent (`France`) | ⚠️ Bruyant |
| E | Paraphrase | ❌ Faible → Qdrant module 11 |
| F | Terme absent | ✅ 0 hit (correct) |
| G | Filtre `--lang fr/en` | ✅ Corpus natif (ADR 0003) |

**Score** : plus haut = plus pertinent *pour cette requête* ; comparer seulement à l'intérieur d'une même recherche.

CLI équivalente : `uv run presslake search "Népal" --lang fr`